This notebook builds the modeling pipeline across all six feature sets (`01_set_a.csv` ~ `06_set_e2.csv`). Since the goal is a model that keeps predicting after deployment, not just an analysis of historical data, the train/test split is time-based, with systematic safeguards from data leakage.

**Scope of This Notebook**

| Step | What happens | Data leakage safeguard |
|---|---|---|
| Load & align sets | Load all six sets, sanity-check they agree on shared columns | `row_id`, `Review Date`, `Date Flown` excluded from the feature matrix |
| Train/Test split | **Chronological** split by `Review Date` (8:2) applied to all sets | Test = genuinely future data the model never sees |
| Set A imputation | Re-validate Type Of Traveller vs Seat Type on the **train split only**, then fit a group-median imputer on train only | Statistics learned from train rows only |
| Scaling | `StandardScaler`, train-fit, LR branch only | Fit on train only, inside the pipeline |
| Encoding | One-Hot Encode categoricals, fit on train only | `OneHotEncoder(handle_unknown='ignore')` |
| Modeling pipeline | `sklearn.Pipeline` per (Set x Model) | Every preprocessing step lives inside the pipeline |
| Models | Logistic Regression, Random Forest, XGBoost, LightGBM | `class_weight='balanced'` / `scale_pos_weight`, computed from train only |
| Evaluation | `TimeSeriesSplit` (5 folds, forward-chaining) on train + final test | CV folds also respect chronological order |
| Set x Model comparison | Preliminary matrix + paired significance test | Confirms whether the Set ranking holds across all four model families |

**Why a chronological split** 

This study aims to generalize to future reviews. A random split doesn't actually test that as it lets future-period reviews leak into training and evaluates on a mix of the same era. A chronological split is the honest test of trained on the past, does it hold up on the future.

**Why scale only for Logistic Regression**

Logistic Regression's regularization is scale-sensitive, so features need comparable ranges. Tree-based models (RF/XGBoost/LightGBM) are scale-invariant by construction, so scaling them would add compute with no effect on results.

## **1. Setup**

GroupMedianImputer, build_pipeline, SET_SCHEMA and related logic live in `modeling_utils.py` instead of being redefined inline in each notebook. This matters because `08_hyperparameter_tuning.ipynb` saves fitted pipelines with `joblib`, and `09_shap_analysis.ipynb` loads them in a separate kernel. As `joblib` can only unpickle a custom class if it can re-import it from the same path used to save it. A class defined inline in a notebook has no such path once that notebook's kernel closes. A real module gives every notebook the same, always-importable path.

In [1]:
# If not already installed in this environment: !pip install xgboost lightgbm --break-system-packages -q

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

from modelling_utils import (
    RANDOM_STATE, TARGET, CAT_COLS, ID_COL, NON_FEATURE_COLS, RATING_COLS,
    SET_SCHEMA, GroupMedianImputer, clean_numeric_cols_for, get_model,
    build_pipeline, load_sets_and_split,
)

DATA_DIR = "../1_data/02_final_sets/"


## **2. Load All Sets & Confirm Alignment**

In [2]:
SET_FILES = {
    "A": "01_set_a.csv",
    "B": "02_set_b.csv",
    "C": "03_set_c.csv",
    "D": "04_set_d.csv",
    "E1": "05_set_e1.csv",
    "E2": "06_set_e2.csv",
}

sets = {name: pd.read_csv(DATA_DIR + fname, parse_dates=["Review Date", "Date Flown"])
        for name, fname in SET_FILES.items()}

print("Row counts:", {name: len(df) for name, df in sets.items()})

ref = sets["A"][[ID_COL, "Recommended", "review_length", "Verified"]].sort_values(ID_COL).reset_index(drop=True)
for name, df in sets.items():
    check = df[[ID_COL, "Recommended", "review_length", "Verified"]].sort_values(ID_COL).reset_index(drop=True)
    aligned = (check == ref).all().all()
    print(f"  Set {name} aligned with Set A (by row_id): {aligned}")
    assert aligned, f"Set {name} does not match Set A on shared columns for the same row_id"


Row counts: {'A': 22980, 'B': 22980, 'C': 22980, 'D': 22980, 'E1': 22980, 'E2': 22980}
  Set A aligned with Set A (by row_id): True
  Set B aligned with Set A (by row_id): True
  Set C aligned with Set A (by row_id): True
  Set D aligned with Set A (by row_id): True
  Set E1 aligned with Set A (by row_id): True
  Set E2 aligned with Set A (by row_id): True


## **3. Train/Test Split**

Sort by `Review Date` rather than `Date Flown`, which has 15.9% missing values. Because this is a chronological split, the class balance can differ between train and test. In fact, this is reported below rather than corrected, since that shift is itself part of what a chronological split is meant to reveal. This is kept in mind when interpreting results: metrics sensitive to class prevalence (Accuracy, F1) are read with more caution than `ROC-AUC`, which is comparatively robust to this shift.

In [ ]:
sorted_by_date = sets["A"].sort_values("Review Date").reset_index(drop=True)
target_sorted = sorted_by_date[TARGET] 

n = len(sorted_by_date)
cut = int(n * 0.8)
...
print(f"Train Recommended rate: {target_sorted.iloc[:cut].mean():.3f}")
print(f"Test Recommended rate:  {target_sorted.iloc[cut:].mean():.3f}")

Train Recommended rate: 0.359
Test Recommended rate:  0.235
